# Side Effect Primary Analysis

Recovered workflow. See `docs/WORKFLOW_ORDER.md` for dependencies and execution restrictions. Source cell numbers below are zero-based.


Source cell: 29.


In [ ]:
# ============================================================
# FINAL UNIQUE263 SIDE-EFFECT ANALYSIS
# PRIMARY = ORIGINAL EQUAL 3-SEED LGBM ENSEMBLE
# ============================================================
#
# NO TRAINING
# NO HYBRID
# NO TEST-BASED THRESHOLD SELECTION
#
# Analyses:
# 1) Per-label prevalence + Precision/Recall/F1 for 263 labels
# 2) Rare / Medium / Frequent
# 3) Threshold comparison
# 4) Threshold sensitivity
# 5) Calibration: Brier / ECE / NLL
# 6) Special labels: anaphylaxis / hyperglycemia / tinnitus
# 7) Excel + CSV + figures + ZIP
# ============================================================


# ============================================================
# 0) IMPORTS
# ============================================================

from pathlib import Path
import json
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    precision_recall_fscore_support
)

warnings.filterwarnings("ignore")


# ============================================================
# 1) FIXED PATHS
# ============================================================

DATA_ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-unique263-cal-test-stage2"
)

BASELINE_ROOT = (
    DATA_ROOT /
    "PRIMARY_UNIQUE263_BASELINE_PROBS"
)

# FINAL TARGET MATRICES
Y_TRAIN_PATH = DATA_ROOT / "Y_train (1).npy"
Y_CAL_PATH   = DATA_ROOT / "Y_calibration.npy"
Y_TEST_PATH  = DATA_ROOT / "Y_test.npy"

LABEL_MAP_PATH = DATA_ROOT / "label_mapping.csv"


# OUTPUT
OUT = Path(
    "/kaggle/working/"
    "FINAL_UNIQUE263_PRIMARY_SIDE_EFFECT_ANALYSIS"
)

if OUT.exists():
    shutil.rmtree(OUT)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2) CONFIGURATION
# ============================================================

SEEDS = [42, 43, 44]

N_LABELS = 263

EXPECTED_TRAIN_N = 134249
EXPECTED_CAL_N = 28768
EXPECTED_TEST_N = 28768

MIN_CAL_SUPPORT = 3

ECE_BINS = 15


GLOBAL_GRID = np.round(
    np.arange(
        0.05,
        0.851,
        0.01
    ),
    2
)


PER_LABEL_GRID = np.round(
    np.arange(
        0.01,
        0.951,
        0.01
    ),
    2
)


# ============================================================
# 3) PATH CHECK
# ============================================================

print("=" * 120)
print("PATH CHECK")
print("=" * 120)

paths_to_check = {

    "DATA_ROOT":
        DATA_ROOT,

    "BASELINE_ROOT":
        BASELINE_ROOT,

    "Y_train":
        Y_TRAIN_PATH,

    "Y_calibration":
        Y_CAL_PATH,

    "Y_test":
        Y_TEST_PATH,

    "label_mapping":
        LABEL_MAP_PATH
}


for name, path in paths_to_check.items():

    print(
        f"{name:18s}: "
        f"{path}"
    )

    print(
        f"{'':18s}  "
        f"Exists = {path.exists()}"
    )


assert DATA_ROOT.exists()
assert BASELINE_ROOT.exists()
assert Y_TRAIN_PATH.exists()
assert Y_CAL_PATH.exists()
assert Y_TEST_PATH.exists()
assert LABEL_MAP_PATH.exists()


print(
    "\n All required paths exist"
)


# ============================================================
# 4) VERIFY ORIGINAL BASELINE PROBABILITIES
# ============================================================

print("\n" + "=" * 120)
print("VERIFYING ORIGINAL PRIMARY BASELINE")
print("=" * 120)


seed_dirs = {}


for seed in SEEDS:

    seed_dir = (
        BASELINE_ROOT /
        f"seed_{seed}"
    )

    cal_prob_path = (
        seed_dir /
        "calibration_probabilities.npy"
    )

    test_prob_path = (
        seed_dir /
        "test_probabilities.npy"
    )


    print(
        f"\nSeed {seed}"
    )

    print(
        "Folder:",
        seed_dir
    )

    print(
        "Calibration exists:",
        cal_prob_path.exists()
    )

    print(
        "Test exists:",
        test_prob_path.exists()
    )


    assert cal_prob_path.exists()
    assert test_prob_path.exists()


    cal_tmp = np.load(
        cal_prob_path,
        mmap_mode="r"
    )

    test_tmp = np.load(
        test_prob_path,
        mmap_mode="r"
    )


    print(
        "Calibration:",
        cal_tmp.shape,
        cal_tmp.dtype
    )

    print(
        "Test:",
        test_tmp.shape,
        test_tmp.dtype
    )


    assert cal_tmp.shape == (
        EXPECTED_CAL_N,
        N_LABELS
    )

    assert test_tmp.shape == (
        EXPECTED_TEST_N,
        N_LABELS
    )


    assert "hybrid" not in str(
        seed_dir
    ).lower()


    seed_dirs[
        seed
    ] = seed_dir


print(
    "\n ORIGINAL BASELINE confirmed"
)

print(
    " Seeds 42 / 43 / 44"
)

print(
    " No HYBRID"
)

print(
    " No training"
)


# ============================================================
# 5) LOAD TARGET MATRICES
# ============================================================

print("\n" + "=" * 120)
print("LOADING FINAL UNIQUE263 TARGET MATRICES")
print("=" * 120)


Y_train = np.load(
    Y_TRAIN_PATH
).astype(
    np.uint8
)


Y_cal = np.load(
    Y_CAL_PATH
).astype(
    np.uint8
)


Y_test = np.load(
    Y_TEST_PATH
).astype(
    np.uint8
)


print(
    "Y_train:",
    Y_train.shape,
    Y_train.dtype
)

print(
    "Y_cal:",
    Y_cal.shape,
    Y_cal.dtype
)

print(
    "Y_test:",
    Y_test.shape,
    Y_test.dtype
)


assert Y_train.shape == (
    EXPECTED_TRAIN_N,
    N_LABELS
)

assert Y_cal.shape == (
    EXPECTED_CAL_N,
    N_LABELS
)

assert Y_test.shape == (
    EXPECTED_TEST_N,
    N_LABELS
)


for name, arr in [
    ("Y_train", Y_train),
    ("Y_cal", Y_cal),
    ("Y_test", Y_test)
]:

    unique_vals = np.unique(
        arr
    )

    print(
        name,
        "unique values:",
        unique_vals
    )

    assert np.isin(
        unique_vals,
        [0, 1]
    ).all()


print(
    "\n All target matrices valid"
)


# ============================================================
# 6) LOAD LABEL MAPPING
# ============================================================

label_map = pd.read_csv(
    LABEL_MAP_PATH
)


print("\nLabel mapping columns:")
print(
    label_map.columns.tolist()
)


assert "label_id" in label_map.columns
assert "label" in label_map.columns


label_map = (
    label_map
    .sort_values(
        "label_id"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nLabel mapping shape:",
    label_map.shape
)


assert len(
    label_map
) == N_LABELS


assert label_map[
    "label_id"
].nunique() == N_LABELS


assert label_map[
    "label"
].nunique() == N_LABELS


assert np.array_equal(

    label_map[
        "label_id"
    ].astype(int).values,

    np.arange(
        N_LABELS
    )
)


# ============================================================
# 7) SPECIAL LABEL QA
# ============================================================

SPECIAL_LABELS = {

    5:
        "anaphylaxis",

    91:
        "hyperglycemia",

    254:
        "tinnitus"
}


print("\n" + "=" * 120)
print("SPECIAL LABEL QA")
print("=" * 120)


for label_id, expected_label in (
    SPECIAL_LABELS.items()
):

    actual_label = str(

        label_map.loc[

            label_map[
                "label_id"
            ]
            ==
            label_id,

            "label"

        ].iloc[0]
    )


    print(
        label_id,
        "→",
        actual_label
    )


    assert (
        actual_label
        ==
        expected_label
    )


print(
    "\n Final 263-label mapping confirmed"
)


# ============================================================
# 8) LOAD THREE SEED PROBABILITIES
# ============================================================

print("\n" + "=" * 120)
print("LOADING PRIMARY 3-SEED PROBABILITIES")
print("=" * 120)


cal_probs = []
test_probs = []


for seed in SEEDS:

    seed_dir = (
        seed_dirs[
            seed
        ]
    )


    cal_p = np.load(
        seed_dir /
        "calibration_probabilities.npy"
    ).astype(
        np.float32
    )


    test_p = np.load(
        seed_dir /
        "test_probabilities.npy"
    ).astype(
        np.float32
    )


    print(
        f"Seed {seed}: "
        f"cal={cal_p.shape}, "
        f"test={test_p.shape}"
    )


    assert cal_p.shape == Y_cal.shape
    assert test_p.shape == Y_test.shape


    assert np.isfinite(
        cal_p
    ).all()


    assert np.isfinite(
        test_p
    ).all()


    assert (
        cal_p.min()
        >=
        0
    )


    assert (
        cal_p.max()
        <=
        1
    )


    assert (
        test_p.min()
        >=
        0
    )


    assert (
        test_p.max()
        <=
        1
    )


    cal_probs.append(
        cal_p
    )


    test_probs.append(
        test_p
    )


# ============================================================
# 9) BUILD EQUAL 3-SEED ENSEMBLE
# ============================================================

P_cal = np.mean(

    np.stack(
        cal_probs,
        axis=0
    ),

    axis=0

).astype(
    np.float32
)


P_test = np.mean(

    np.stack(
        test_probs,
        axis=0
    ),

    axis=0

).astype(
    np.float32
)


print(
    "\nP_cal:",
    P_cal.shape
)

print(
    "P_test:",
    P_test.shape
)


print(
    "\n Equal arithmetic mean ensemble built"
)


# ============================================================
# 10) FAST THRESHOLD COUNTS
# ============================================================

def threshold_counts(
    P,
    Y,
    thresholds
):

    thresholds = np.asarray(
        thresholds,
        dtype=float
    )


    n_samples, n_labels = (
        P.shape
    )


    n_t = len(
        thresholds
    )


    TP = np.zeros(
        (
            n_labels,
            n_t
        ),
        dtype=np.int64
    )


    FP = np.zeros_like(
        TP
    )


    FN = np.zeros_like(
        TP
    )


    support = (
        Y.sum(
            axis=0
        )
        .astype(
            np.int64
        )
    )


    for j in range(
        n_labels
    ):

        p = P[
            :,
            j
        ]


        y = (
            Y[
                :,
                j
            ]
            .astype(
                np.int8
            )
        )


        order = np.argsort(
            -p
        )


        ps = p[
            order
        ]


        ys = y[
            order
        ]


        cum_tp = np.cumsum(
            ys,
            dtype=np.int64
        )


        k = np.searchsorted(
            -ps,
            -thresholds,
            side="right"
        )


        tp = np.zeros(
            n_t,
            dtype=np.int64
        )


        mask = (
            k > 0
        )


        tp[
            mask
        ] = (
            cum_tp[
                k[
                    mask
                ]
                -
                1
            ]
        )


        fp = (
            k.astype(
                np.int64
            )
            -
            tp
        )


        fn = (
            support[
                j
            ]
            -
            tp
        )


        TP[
            j
        ] = tp


        FP[
            j
        ] = fp


        FN[
            j
        ] = fn


    return (
        TP,
        FP,
        FN
    )


# ============================================================
# 11) F1 FROM COUNTS
# ============================================================

def f1_from_counts(
    tp,
    fp,
    fn
):

    tp = np.asarray(
        tp
    )

    fp = np.asarray(
        fp
    )

    fn = np.asarray(
        fn
    )


    denom = (
        2 * tp
        +
        fp
        +
        fn
    )


    result = np.zeros(
        np.shape(
            denom
        ),
        dtype=float
    )


    mask = (
        denom > 0
    )


    result[
        mask
    ] = (

        2
        *
        tp[
            mask
        ]
        /
        denom[
            mask
        ]
    )


    return result


# ============================================================
# 12) GLOBAL THRESHOLD — CALIBRATION ONLY
# ============================================================

print("\n" + "=" * 120)
print("GLOBAL THRESHOLD SELECTION — CALIBRATION ONLY")
print("=" * 120)


TP_global, FP_global, FN_global = (
    threshold_counts(
        P_cal,
        Y_cal,
        GLOBAL_GRID
    )
)


global_label_f1 = (
    f1_from_counts(
        TP_global,
        FP_global,
        FN_global
    )
)


global_macro_f1 = (
    global_label_f1.mean(
        axis=0
    )
)


global_micro_f1 = (
    f1_from_counts(

        TP_global.sum(
            axis=0
        ),

        FP_global.sum(
            axis=0
        ),

        FN_global.sum(
            axis=0
        )
    )
)


global_objective = (

    0.5
    *
    global_micro_f1

    +

    0.5
    *
    global_macro_f1
)


best_global_idx = int(
    np.argmax(
        global_objective
    )
)


BEST_GLOBAL_THRESHOLD = float(
    GLOBAL_GRID[
        best_global_idx
    ]
)


print(
    "Best global threshold:",
    BEST_GLOBAL_THRESHOLD
)


print(
    "Calibration Micro-F1:",
    global_micro_f1[
        best_global_idx
    ]
)


print(
    "Calibration Macro-F1:",
    global_macro_f1[
        best_global_idx
    ]
)


print(
    "Objective:",
    global_objective[
        best_global_idx
    ]
)


# ============================================================
# 13) PER-LABEL THRESHOLDS — CALIBRATION ONLY
# ============================================================

print("\n" + "=" * 120)
print("PER-LABEL THRESHOLD SELECTION — CALIBRATION ONLY")
print("=" * 120)


TP_label, FP_label, FN_label = (
    threshold_counts(
        P_cal,
        Y_cal,
        PER_LABEL_GRID
    )
)


per_label_grid_f1 = (
    f1_from_counts(
        TP_label,
        FP_label,
        FN_label
    )
)


cal_support = (
    Y_cal.sum(
        axis=0
    )
    .astype(int)
)


per_label_thresholds = np.full(
    N_LABELS,
    BEST_GLOBAL_THRESHOLD,
    dtype=float
)


for j in range(
    N_LABELS
):

    if (
        cal_support[
            j
        ]
        <
        MIN_CAL_SUPPORT
    ):

        continue


    best_idx = int(
        np.argmax(
            per_label_grid_f1[
                j
            ]
        )
    )


    per_label_thresholds[
        j
    ] = float(
        PER_LABEL_GRID[
            best_idx
        ]
    )


print(
    "Threshold mean:",
    per_label_thresholds.mean()
)

print(
    "Threshold median:",
    np.median(
        per_label_thresholds
    )
)

print(
    "Threshold min:",
    per_label_thresholds.min()
)

print(
    "Threshold max:",
    per_label_thresholds.max()
)


# ============================================================
# 14) APPLY THRESHOLD
# ============================================================

def apply_thresholds(
    P,
    thresholds
):

    if np.isscalar(
        thresholds
    ):

        return (

            P
            >=
            float(
                thresholds
            )

        ).astype(
            np.uint8
        )


    thresholds = np.asarray(
        thresholds
    )


    return (

        P
        >=
        thresholds[
            None,
            :
        ]

    ).astype(
        np.uint8
    )


# ============================================================
# 15) MULTILABEL METRICS
# ============================================================

def multilabel_metrics(
    Y,
    pred
):

    return {

        "Exact_Match":
            accuracy_score(
                Y,
                pred
            ),

        "Micro_Precision":
            precision_score(
                Y,
                pred,
                average="micro",
                zero_division=0
            ),

        "Micro_Recall":
            recall_score(
                Y,
                pred,
                average="micro",
                zero_division=0
            ),

        "Micro_F1":
            f1_score(
                Y,
                pred,
                average="micro",
                zero_division=0
            ),

        "Macro_Precision":
            precision_score(
                Y,
                pred,
                average="macro",
                zero_division=0
            ),

        "Macro_Recall":
            recall_score(
                Y,
                pred,
                average="macro",
                zero_division=0
            ),

        "Macro_F1":
            f1_score(
                Y,
                pred,
                average="macro",
                zero_division=0
            ),

        "Weighted_F1":
            f1_score(
                Y,
                pred,
                average="weighted",
                zero_division=0
            ),

        "Hamming_Loss":
            hamming_loss(
                Y,
                pred
            )
    }


# ============================================================
# 16) SELECT GLOBAL VS PER-LABEL USING CALIBRATION ONLY
# ============================================================

pred_cal_global = apply_thresholds(
    P_cal,
    BEST_GLOBAL_THRESHOLD
)


pred_cal_perlabel = apply_thresholds(
    P_cal,
    per_label_thresholds
)


cal_global_metrics = multilabel_metrics(
    Y_cal,
    pred_cal_global
)


cal_perlabel_metrics = multilabel_metrics(
    Y_cal,
    pred_cal_perlabel
)


cal_global_objective = (

    0.5
    *
    cal_global_metrics[
        "Micro_F1"
    ]

    +

    0.5
    *
    cal_global_metrics[
        "Macro_F1"
    ]
)


cal_perlabel_objective = (

    0.5
    *
    cal_perlabel_metrics[
        "Micro_F1"
    ]

    +

    0.5
    *
    cal_perlabel_metrics[
        "Macro_F1"
    ]
)


if (
    cal_perlabel_objective
    >
    cal_global_objective
):

    SELECTED_METHOD = (
        "PER_LABEL"
    )

    SELECTED_THRESHOLDS = (
        per_label_thresholds
    )

else:

    SELECTED_METHOD = (
        "GLOBAL"
    )

    SELECTED_THRESHOLDS = (
        BEST_GLOBAL_THRESHOLD
    )


print("\n" + "=" * 120)
print("CALIBRATION METHOD SELECTION")
print("=" * 120)


print(
    "Global objective:",
    cal_global_objective
)


print(
    "Per-label objective:",
    cal_perlabel_objective
)


print(
    "Selected method:",
    SELECTED_METHOD
)


# ============================================================
# 17) FINAL TEST PREDICTIONS
# ============================================================

pred_test = apply_thresholds(
    P_test,
    SELECTED_THRESHOLDS
)


primary_test_metrics = (
    multilabel_metrics(
        Y_test,
        pred_test
    )
)


print("\n" + "=" * 120)
print("PRIMARY FINAL TEST METRICS")
print("=" * 120)


for key, value in (
    primary_test_metrics.items()
):

    if key == "Hamming_Loss":

        print(
            f"{key:22s}: "
            f"{value:.8f}"
        )

    else:

        print(
            f"{key:22s}: "
            f"{100*value:.4f}%"
        )


# ============================================================
# 18) PRIMARY MODEL SIGNATURE
# ============================================================

signature_df = pd.DataFrame({

    "Check": [

        "Best global threshold approximately 0.54",

        "Per-label calibration objective approximately 0.927551",

        "Selected threshold method PER_LABEL",

        "Test Micro-F1 approximately 0.916328",

        "Test Macro-F1 approximately 0.920570"
    ],

    "Observed": [

        BEST_GLOBAL_THRESHOLD,

        cal_perlabel_objective,

        SELECTED_METHOD,

        primary_test_metrics[
            "Micro_F1"
        ],

        primary_test_metrics[
            "Macro_F1"
        ]
    ],

    "Passed": [

        bool(
            abs(
                BEST_GLOBAL_THRESHOLD
                -
                0.54
            )
            <
            0.02
        ),

        bool(
            abs(
                cal_perlabel_objective
                -
                0.927551
            )
            <
            0.002
        ),

        bool(
            SELECTED_METHOD
            ==
            "PER_LABEL"
        ),

        bool(
            abs(
                primary_test_metrics[
                    "Micro_F1"
                ]
                -
                0.916328
            )
            <
            0.002
        ),

        bool(
            abs(
                primary_test_metrics[
                    "Macro_F1"
                ]
                -
                0.920570
            )
            <
            0.002
        )
    ]
})


print("\n" + "=" * 120)
print("PRIMARY MODEL SIGNATURE")
print("=" * 120)


print(
    signature_df.to_string(
        index=False
    )
)


# ============================================================
# 19) TRAIN / CAL / TEST PREVALENCE
# ============================================================

train_support = (
    Y_train.sum(
        axis=0
    )
    .astype(int)
)


cal_support = (
    Y_cal.sum(
        axis=0
    )
    .astype(int)
)


test_support = (
    Y_test.sum(
        axis=0
    )
    .astype(int)
)


# ============================================================
# 20) FREQUENCY GROUPS
# ============================================================

def frequency_group(
    support
):

    if support < 100:

        return (
            "Rare (<100)"
        )

    elif support < 1000:

        return (
            "Medium (100-999)"
        )

    else:

        return (
            "Frequent (>=1000)"
        )


frequency_groups = [

    frequency_group(
        x
    )

    for x in train_support
]


# ============================================================
# 21) PER-LABEL TEST METRICS
# ============================================================

(
    precision_l,
    recall_l,
    f1_l,
    support_from_metric
) = precision_recall_fscore_support(

    Y_test,

    pred_test,

    average=None,

    zero_division=0
)


TP = np.logical_and(
    Y_test == 1,
    pred_test == 1
).sum(
    axis=0
)


FP = np.logical_and(
    Y_test == 0,
    pred_test == 1
).sum(
    axis=0
)


FN = np.logical_and(
    Y_test == 1,
    pred_test == 0
).sum(
    axis=0
)


TN = (
    Y_test.shape[0]
    -
    TP
    -
    FP
    -
    FN
)


specificity = np.divide(

    TN,

    TN + FP,

    out=np.zeros(
        N_LABELS,
        dtype=float
    ),

    where=(
        TN + FP
    )
    >
    0
)


predicted_positive = (
    pred_test.sum(
        axis=0
    )
)


# ============================================================
# 22) CALIBRATION HELPERS
# ============================================================

def binary_nll(
    Y,
    P
):

    eps = 1e-7


    P = np.clip(
        P,
        eps,
        1 - eps
    )


    return float(

        -np.mean(

            Y
            *
            np.log(
                P
            )

            +

            (
                1 - Y
            )
            *
            np.log(
                1 - P
            )
        )
    )


def reliability_bins(
    Y,
    P,
    n_bins=15
):

    y = np.asarray(
        Y
    ).reshape(-1)


    p = np.asarray(
        P
    ).reshape(-1)


    bin_ids = np.floor(
        p
        *
        n_bins
    ).astype(
        np.int16
    )


    bin_ids = np.clip(
        bin_ids,
        0,
        n_bins - 1
    )


    counts = np.bincount(
        bin_ids,
        minlength=n_bins
    )


    probability_sum = np.bincount(
        bin_ids,
        weights=p,
        minlength=n_bins
    )


    positive_sum = np.bincount(
        bin_ids,
        weights=y,
        minlength=n_bins
    )


    mean_probability = np.divide(

        probability_sum,

        counts,

        out=np.zeros(
            n_bins,
            dtype=float
        ),

        where=counts > 0
    )


    observed_frequency = np.divide(

        positive_sum,

        counts,

        out=np.zeros(
            n_bins,
            dtype=float
        ),

        where=counts > 0
    )


    total = counts.sum()


    ece = float(

        np.sum(

            (
                counts
                /
                total
            )

            *

            np.abs(
                mean_probability
                -
                observed_frequency
            )
        )
    )


    table = pd.DataFrame({

        "Bin":
            np.arange(
                n_bins
            ),

        "Lower":
            np.arange(
                n_bins
            )
            /
            n_bins,

        "Upper":
            (
                np.arange(
                    n_bins
                )
                +
                1
            )
            /
            n_bins,

        "Count":
            counts,

        "Mean_Predicted_Probability":
            mean_probability,

        "Observed_Frequency":
            observed_frequency
    })


    return (
        ece,
        table
    )


# ============================================================
# 23) PER-LABEL CALIBRATION
# ============================================================

brier_per_label = (

    (
        P_test
        -
        Y_test
    )
    **
    2

).mean(
    axis=0
)


nll_per_label = np.zeros(
    N_LABELS,
    dtype=float
)


ece_per_label = np.zeros(
    N_LABELS,
    dtype=float
)


for j in range(
    N_LABELS
):

    nll_per_label[
        j
    ] = binary_nll(

        Y_test[
            :,
            j
        ],

        P_test[
            :,
            j
        ]
    )


    ece_per_label[
        j
    ], _ = reliability_bins(

        Y_test[
            :,
            j
        ],

        P_test[
            :,
            j
        ],

        ECE_BINS
    )


# ============================================================
# 24) FINAL 263-LABEL TABLE
# ============================================================

if (
    SELECTED_METHOD
    ==
    "PER_LABEL"
):

    threshold_vector = (
        per_label_thresholds
    )

else:

    threshold_vector = np.full(
        N_LABELS,
        BEST_GLOBAL_THRESHOLD
    )


per_label = pd.DataFrame({

    "label_id":
        label_map[
            "label_id"
        ].astype(int),

    "label":
        label_map[
            "label"
        ].astype(str),

    "Train_Support":
        train_support,

    "Train_Prevalence":
        train_support
        /
        Y_train.shape[0],

    "Calibration_Support":
        cal_support,

    "Calibration_Prevalence":
        cal_support
        /
        Y_cal.shape[0],

    "Test_Support":
        test_support,

    "Test_Prevalence":
        test_support
        /
        Y_test.shape[0],

    "Frequency_Group":
        frequency_groups,

    "Selected_Threshold":
        threshold_vector,

    "Predicted_Positive":
        predicted_positive,

    "TP":
        TP,

    "FP":
        FP,

    "FN":
        FN,

    "TN":
        TN,

    "Precision":
        precision_l,

    "Recall":
        recall_l,

    "F1":
        f1_l,

    "Specificity":
        specificity,

    "Brier_Score":
        brier_per_label,

    "ECE_15bin":
        ece_per_label,

    "NLL":
        nll_per_label
})


print("\n" + "=" * 120)
print("PER-LABEL TABLE QA")
print("=" * 120)

print(
    "Rows:",
    len(
        per_label
    )
)

assert len(
    per_label
) == 263


print(
    per_label.head(
        10
    ).to_string(
        index=False
    )
)


# ============================================================
# 25) FREQUENCY GROUP ANALYSIS
# ============================================================

def calculate_frequency_group(
    group_name
):

    idx = np.where(

        per_label[
            "Frequency_Group"
        ].values
        ==
        group_name

    )[0]


    if len(
        idx
    ) == 0:

        return {

            "Frequency_Group":
                group_name,

            "Number_of_Labels":
                0
        }


    y = Y_test[
        :,
        idx
    ]


    pred = pred_test[
        :,
        idx
    ]


    probs = P_test[
        :,
        idx
    ]


    m = multilabel_metrics(
        y,
        pred
    )


    group_ece, _ = reliability_bins(
        y,
        probs,
        ECE_BINS
    )


    return {

        "Frequency_Group":
            group_name,

        "Number_of_Labels":
            len(
                idx
            ),

        "Total_Train_Positive_Support":
            int(
                train_support[
                    idx
                ].sum()
            ),

        "Total_Test_Positive_Support":
            int(
                y.sum()
            ),

        "Test_Cell_Prevalence":
            float(
                y.mean()
            ),

        "Micro_Precision":
            m[
                "Micro_Precision"
            ],

        "Micro_Recall":
            m[
                "Micro_Recall"
            ],

        "Micro_F1":
            m[
                "Micro_F1"
            ],

        "Macro_Precision":
            m[
                "Macro_Precision"
            ],

        "Macro_Recall":
            m[
                "Macro_Recall"
            ],

        "Macro_F1":
            m[
                "Macro_F1"
            ],

        "Weighted_F1":
            m[
                "Weighted_F1"
            ],

        "Mean_Brier_Score":
            float(
                brier_per_label[
                    idx
                ].mean()
            ),

        "Mean_Label_ECE":
            float(
                ece_per_label[
                    idx
                ].mean()
            ),

        "Micro_ECE":
            group_ece
    }


frequency_summary = pd.DataFrame(
    [

        calculate_frequency_group(
            "Rare (<100)"
        ),

        calculate_frequency_group(
            "Medium (100-999)"
        ),

        calculate_frequency_group(
            "Frequent (>=1000)"
        )
    ]
)


# ============================================================
# 26) THRESHOLD COMPARISON
# ============================================================

threshold_strategies = {

    "Fixed_0.50":
        0.50,

    "Calibration_Global":
        BEST_GLOBAL_THRESHOLD,

    "Calibration_PerLabel":
        per_label_thresholds
}


threshold_rows = []


for strategy_name, thresholds in (
    threshold_strategies.items()
):

    cal_pred = apply_thresholds(
        P_cal,
        thresholds
    )


    test_pred = apply_thresholds(
        P_test,
        thresholds
    )


    cal_m = multilabel_metrics(
        Y_cal,
        cal_pred
    )


    test_m = multilabel_metrics(
        Y_test,
        test_pred
    )


    threshold_rows.append({

        "Threshold_Strategy":
            strategy_name,

        "Calibration_Micro_F1":
            cal_m[
                "Micro_F1"
            ],

        "Calibration_Macro_F1":
            cal_m[
                "Macro_F1"
            ],

        "Calibration_Objective":
            0.5
            *
            (
                cal_m[
                    "Micro_F1"
                ]
                +
                cal_m[
                    "Macro_F1"
                ]
            ),

        "Test_Exact_Match":
            test_m[
                "Exact_Match"
            ],

        "Test_Micro_Precision":
            test_m[
                "Micro_Precision"
            ],

        "Test_Micro_Recall":
            test_m[
                "Micro_Recall"
            ],

        "Test_Micro_F1":
            test_m[
                "Micro_F1"
            ],

        "Test_Macro_Precision":
            test_m[
                "Macro_Precision"
            ],

        "Test_Macro_Recall":
            test_m[
                "Macro_Recall"
            ],

        "Test_Macro_F1":
            test_m[
                "Macro_F1"
            ],

        "Test_Weighted_F1":
            test_m[
                "Weighted_F1"
            ],

        "Test_Hamming_Loss":
            test_m[
                "Hamming_Loss"
            ]
    })


threshold_comparison = pd.DataFrame(
    threshold_rows
)


# ============================================================
# 27) GLOBAL THRESHOLD SENSITIVITY
# ============================================================

TP_test_grid, FP_test_grid, FN_test_grid = (
    threshold_counts(
        P_test,
        Y_test,
        GLOBAL_GRID
    )
)


test_label_f1_grid = (
    f1_from_counts(
        TP_test_grid,
        FP_test_grid,
        FN_test_grid
    )
)


test_macro_grid = (
    test_label_f1_grid.mean(
        axis=0
    )
)


test_micro_grid = (
    f1_from_counts(

        TP_test_grid.sum(
            axis=0
        ),

        FP_test_grid.sum(
            axis=0
        ),

        FN_test_grid.sum(
            axis=0
        )
    )
)


threshold_sensitivity = pd.DataFrame({

    "Global_Threshold":
        GLOBAL_GRID,

    "Calibration_Micro_F1":
        global_micro_f1,

    "Calibration_Macro_F1":
        global_macro_f1,

    "Calibration_Objective":
        global_objective,

    "Test_Micro_F1_Descriptive":
        test_micro_grid,

    "Test_Macro_F1_Descriptive":
        test_macro_grid,

    "Calibration_Selected":
        GLOBAL_GRID
        ==
        BEST_GLOBAL_THRESHOLD
})


# ============================================================
# 28) PER-LABEL THRESHOLD PERTURBATION
# ============================================================

perturbation_rows = []


for delta in [

    -0.10,
    -0.05,
    0.00,
    +0.05,
    +0.10
]:

    shifted_thresholds = np.clip(

        per_label_thresholds
        +
        delta,

        0.01,

        0.95
    )


    shifted_pred = apply_thresholds(
        P_test,
        shifted_thresholds
    )


    m = multilabel_metrics(
        Y_test,
        shifted_pred
    )


    perturbation_rows.append({

        "Threshold_Delta":
            delta,

        "Exact_Match":
            m[
                "Exact_Match"
            ],

        "Micro_Precision":
            m[
                "Micro_Precision"
            ],

        "Micro_Recall":
            m[
                "Micro_Recall"
            ],

        "Micro_F1":
            m[
                "Micro_F1"
            ],

        "Macro_F1":
            m[
                "Macro_F1"
            ],

        "Hamming_Loss":
            m[
                "Hamming_Loss"
            ]
    })


per_label_sensitivity = pd.DataFrame(
    perturbation_rows
)


# ============================================================
# 29) THRESHOLD DISTRIBUTION
# ============================================================

threshold_distribution = (

    per_label

    .groupby(
        "Frequency_Group",
        sort=False
    )

    .agg(

        Number_of_Labels=(
            "label_id",
            "count"
        ),

        Threshold_Mean=(
            "Selected_Threshold",
            "mean"
        ),

        Threshold_SD=(
            "Selected_Threshold",
            "std"
        ),

        Threshold_Median=(
            "Selected_Threshold",
            "median"
        ),

        Threshold_Min=(
            "Selected_Threshold",
            "min"
        ),

        Threshold_Max=(
            "Selected_Threshold",
            "max"
        )
    )

    .reset_index()
)


# ============================================================
# 30) OVERALL CALIBRATION
# ============================================================

overall_ece, reliability_overall = (
    reliability_bins(
        Y_test,
        P_test,
        ECE_BINS
    )
)


overall_brier = float(

    np.mean(

        (
            P_test
            -
            Y_test
        )
        **
        2
    )
)


overall_nll = binary_nll(
    Y_test,
    P_test
)


# ============================================================
# 31) CALIBRATION BY FREQUENCY
# ============================================================

calibration_rows = [

    {

        "Group":
            "Overall",

        "Labels":
            N_LABELS,

        "Brier_Score_Micro":
            overall_brier,

        "Mean_Label_Brier":
            float(
                brier_per_label.mean()
            ),

        "ECE_Micro":
            overall_ece,

        "Mean_Label_ECE":
            float(
                ece_per_label.mean()
            ),

        "NLL_Micro":
            overall_nll,

        "Mean_Label_NLL":
            float(
                nll_per_label.mean()
            )
    }
]


reliability_tables = [

    reliability_overall.assign(
        Group="Overall"
    )
]


for group_name in [

    "Rare (<100)",
    "Medium (100-999)",
    "Frequent (>=1000)"
]:

    idx = np.where(

        per_label[
            "Frequency_Group"
        ].values
        ==
        group_name

    )[0]


    if len(
        idx
    ) == 0:

        continue


    y = Y_test[
        :,
        idx
    ]


    p = P_test[
        :,
        idx
    ]


    group_ece, rel_table = (
        reliability_bins(
            y,
            p,
            ECE_BINS
        )
    )


    calibration_rows.append({

        "Group":
            group_name,

        "Labels":
            len(
                idx
            ),

        "Brier_Score_Micro":
            float(
                np.mean(
                    (
                        p
                        -
                        y
                    )
                    **
                    2
                )
            ),

        "Mean_Label_Brier":
            float(
                brier_per_label[
                    idx
                ].mean()
            ),

        "ECE_Micro":
            group_ece,

        "Mean_Label_ECE":
            float(
                ece_per_label[
                    idx
                ].mean()
            ),

        "NLL_Micro":
            binary_nll(
                y,
                p
            ),

        "Mean_Label_NLL":
            float(
                nll_per_label[
                    idx
                ].mean()
            )
    })


    rel_table[
        "Group"
    ] = group_name


    reliability_tables.append(
        rel_table
    )


calibration_summary = pd.DataFrame(
    calibration_rows
)


reliability_all = pd.concat(
    reliability_tables,
    ignore_index=True
)


# ============================================================
# 32) SPECIAL 3
# ============================================================

special3 = (

    per_label[
        per_label[
            "label_id"
        ].isin(
            [
                5,
                91,
                254
            ]
        )
    ]

    .copy()
)


# ============================================================
# 33) LOWEST F1 / HIGHEST SUPPORT
# ============================================================

lowest_f1 = (

    per_label

    .sort_values(
        [
            "F1",
            "Test_Support"
        ],
        ascending=[
            True,
            False
        ]
    )

    .head(
        25
    )
)


highest_support = (

    per_label

    .sort_values(
        "Test_Support",
        ascending=False
    )

    .head(
        25
    )
)


# ============================================================
# 34) SUPPORT VS F1 CORRELATION
# ============================================================

support_f1_spearman = (

    per_label[
        [
            "Train_Support",
            "F1"
        ]
    ]

    .corr(
        method="spearman"
    )

    .iloc[
        0,
        1
    ]
)


correlation_summary = pd.DataFrame({

    "Analysis": [

        "Spearman correlation between training support and test per-label F1"
    ],

    "Value": [

        support_f1_spearman
    ]
})


# ============================================================
# 35) METHOD NOTES
# ============================================================

method_notes = pd.DataFrame({

    "Item": [

        "Primary model",

        "Ensemble",

        "Global threshold selection",

        "Per-label threshold selection",

        "Threshold method selection",

        "Test split",

        "Rare",

        "Medium",

        "Frequent",

        "Calibration",

        "Clinical annotation layer"
    ],

    "Definition": [

        "Original FINAL UNIQUE263 LightGBM baseline model; Stage2 hybrid probabilities are not used.",

        "Equal arithmetic mean of saved probability matrices from seeds 42, 43, and 44.",

        "Selected exclusively on calibration data from thresholds 0.05-0.85 using 0.5*Micro-F1 + 0.5*Macro-F1.",

        "For labels with calibration support >=3, selected exclusively on calibration data from thresholds 0.01-0.95 by maximum per-label F1.",

        "Global and per-label threshold strategies are compared on calibration objective only.",

        "The test split is evaluated only after threshold rules are fixed. Test curves are descriptive sensitivity analyses.",

        "Training support <100 positive examples.",

        "Training support 100-999 positive examples.",

        "Training support >=1000 positive examples.",

        "Brier score, 15-bin expected calibration error, negative log-likelihood, and reliability tables.",

        "Evidence V8, clinical categories, and candidate MedDRA mappings are downstream interpretation layers and do not modify model predictions."
    ]
})


# ============================================================
# 36) FINAL QA
# ============================================================

qa = pd.DataFrame({

    "QA_Check": [

        "Train rows = 134249",

        "Calibration rows = 28768",

        "Test rows = 28768",

        "Labels = 263",

        "Original three seeds loaded",

        "No hybrid folder used",

        "Anaphylaxis ID = 5",

        "Hyperglycemia ID = 91",

        "Tinnitus ID = 254",

        "Threshold selection used calibration only"
    ],

    "Passed": [

        Y_train.shape
        ==
        (
            134249,
            263
        ),

        Y_cal.shape
        ==
        (
            28768,
            263
        ),

        Y_test.shape
        ==
        (
            28768,
            263
        ),

        len(
            label_map
        )
        ==
        263,

        len(
            seed_dirs
        )
        ==
        3,

        all(
            "hybrid"
            not in
            str(
                x
            ).lower()
            for x
            in seed_dirs.values()
        ),

        label_map.loc[
            5,
            "label"
        ]
        ==
        "anaphylaxis",

        label_map.loc[
            91,
            "label"
        ]
        ==
        "hyperglycemia",

        label_map.loc[
            254,
            "label"
        ]
        ==
        "tinnitus",

        True
    ]
})


print("\n" + "=" * 120)
print("FINAL QA")
print("=" * 120)


print(
    qa.to_string(
        index=False
    )
)


assert qa[
    "Passed"
].all()


# ============================================================
# 37) SAVE CSV FILES
# ============================================================

per_label.to_csv(
    OUT /
    "FINAL_Per_Label_263_Metrics.csv",
    index=False
)


frequency_summary.to_csv(
    OUT /
    "FINAL_Frequency_Group_Analysis.csv",
    index=False
)


threshold_comparison.to_csv(
    OUT /
    "FINAL_Threshold_Strategy_Comparison.csv",
    index=False
)


threshold_sensitivity.to_csv(
    OUT /
    "FINAL_Global_Threshold_Sensitivity.csv",
    index=False
)


per_label_sensitivity.to_csv(
    OUT /
    "FINAL_PerLabel_Threshold_Perturbation.csv",
    index=False
)


threshold_distribution.to_csv(
    OUT /
    "FINAL_Threshold_Distribution_By_Frequency.csv",
    index=False
)


calibration_summary.to_csv(
    OUT /
    "FINAL_Calibration_Summary.csv",
    index=False
)


reliability_all.to_csv(
    OUT /
    "FINAL_Reliability_Bins.csv",
    index=False
)


special3.to_csv(
    OUT /
    "FINAL_Special_3_Label_Analysis.csv",
    index=False
)


correlation_summary.to_csv(
    OUT /
    "FINAL_Support_F1_Correlation.csv",
    index=False
)


signature_df.to_csv(
    OUT /
    "FINAL_Primary_Model_Signature.csv",
    index=False
)


qa.to_csv(
    OUT /
    "FINAL_QA.csv",
    index=False
)


# ============================================================
# 38) FIGURE — THRESHOLD SENSITIVITY
# ============================================================

plt.figure(
    figsize=(
        8,
        5
    )
)


plt.plot(

    threshold_sensitivity[
        "Global_Threshold"
    ],

    threshold_sensitivity[
        "Calibration_Micro_F1"
    ],

    label="Calibration Micro-F1"
)


plt.plot(

    threshold_sensitivity[
        "Global_Threshold"
    ],

    threshold_sensitivity[
        "Calibration_Macro_F1"
    ],

    label="Calibration Macro-F1"
)


plt.axvline(

    BEST_GLOBAL_THRESHOLD,

    linestyle="--",

    label=(
        f"Selected = "
        f"{BEST_GLOBAL_THRESHOLD:.2f}"
    )
)


plt.xlabel(
    "Global threshold"
)


plt.ylabel(
    "F1"
)


plt.title(
    "Calibration Threshold Sensitivity"
)


plt.legend()


plt.tight_layout()


plt.savefig(

    OUT /
    "Figure_Threshold_Sensitivity.png",

    dpi=300,

    bbox_inches="tight"
)


plt.close()


# ============================================================
# 39) FIGURE — RELIABILITY
# ============================================================

rel_plot = reliability_overall[
    reliability_overall[
        "Count"
    ]
    >
    0
]


plt.figure(
    figsize=(
        6,
        6
    )
)


plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)


plt.plot(

    rel_plot[
        "Mean_Predicted_Probability"
    ],

    rel_plot[
        "Observed_Frequency"
    ],

    marker="o",

    label="Equal 3-seed ensemble"
)


plt.xlabel(
    "Mean predicted probability"
)


plt.ylabel(
    "Observed positive frequency"
)


plt.title(
    "Reliability Diagram"
)


plt.legend()


plt.tight_layout()


plt.savefig(

    OUT /
    "Figure_Reliability_Diagram.png",

    dpi=300,

    bbox_inches="tight"
)


plt.close()


# ============================================================
# 40) FIGURE — FREQUENCY PERFORMANCE
# ============================================================

plt.figure(
    figsize=(
        8,
        5
    )
)


x = np.arange(
    len(
        frequency_summary
    )
)


width = 0.35


plt.bar(

    x
    -
    width / 2,

    frequency_summary[
        "Micro_F1"
    ],

    width,

    label="Micro-F1"
)


plt.bar(

    x
    +
    width / 2,

    frequency_summary[
        "Macro_F1"
    ],

    width,

    label="Macro-F1"
)


plt.xticks(

    x,

    frequency_summary[
        "Frequency_Group"
    ],

    rotation=10
)


plt.ylabel(
    "F1"
)


plt.title(
    "Side-Effect Performance by Training Frequency"
)


plt.legend()


plt.tight_layout()


plt.savefig(

    OUT /
    "Figure_F1_By_Frequency.png",

    dpi=300,

    bbox_inches="tight"
)


plt.close()


# ============================================================
# 41) FIGURE — SUPPORT VS F1
# ============================================================

plt.figure(
    figsize=(
        8,
        5
    )
)


plt.scatter(

    per_label[
        "Train_Support"
    ],

    per_label[
        "F1"
    ],

    alpha=0.65
)


plt.xscale(
    "log"
)


plt.xlabel(
    "Training positive support (log scale)"
)


plt.ylabel(
    "Test per-label F1"
)


plt.title(
    "Per-Label F1 vs Training Support"
)


plt.tight_layout()


plt.savefig(

    OUT /
    "Figure_PerLabel_F1_vs_Support.png",

    dpi=300,

    bbox_inches="tight"
)


plt.close()


# ============================================================
# 42) EXCEL WORKBOOK
# ============================================================

XLSX_PATH = (
    OUT /
    "FINAL_UNIQUE263_SIDE_EFFECT_ANALYSIS.xlsx"
)


with pd.ExcelWriter(
    XLSX_PATH,
    engine="openpyxl"
) as writer:


    per_label.to_excel(
        writer,
        sheet_name="Per_Label_263",
        index=False
    )


    frequency_summary.to_excel(
        writer,
        sheet_name="Frequency_Groups",
        index=False
    )


    threshold_comparison.to_excel(
        writer,
        sheet_name="Threshold_Comparison",
        index=False
    )


    threshold_sensitivity.to_excel(
        writer,
        sheet_name="Global_Threshold_Sens",
        index=False
    )


    per_label_sensitivity.to_excel(
        writer,
        sheet_name="PerLabel_Threshold_Sens",
        index=False
    )


    threshold_distribution.to_excel(
        writer,
        sheet_name="Threshold_Distribution",
        index=False
    )


    calibration_summary.to_excel(
        writer,
        sheet_name="Calibration_Summary",
        index=False
    )


    reliability_all.to_excel(
        writer,
        sheet_name="Reliability_Bins",
        index=False
    )


    special3.to_excel(
        writer,
        sheet_name="Special_3",
        index=False
    )


    lowest_f1.to_excel(
        writer,
        sheet_name="Lowest_F1_25",
        index=False
    )


    highest_support.to_excel(
        writer,
        sheet_name="Highest_Support_25",
        index=False
    )


    correlation_summary.to_excel(
        writer,
        sheet_name="Support_F1_Correlation",
        index=False
    )


    signature_df.to_excel(
        writer,
        sheet_name="Primary_Model_Signature",
        index=False
    )


    method_notes.to_excel(
        writer,
        sheet_name="Methods_Notes",
        index=False
    )


    qa.to_excel(
        writer,
        sheet_name="QA",
        index=False
    )


# ============================================================
# 43) SAVE SUMMARY JSON
# ============================================================

summary = {

    "model":
        "ORIGINAL_FINAL_UNIQUE263_EQUAL_3SEED_ENSEMBLE",

    "seeds":
        [
            42,
            43,
            44
        ],

    "train_rows":
        int(
            Y_train.shape[
                0
            ]
        ),

    "calibration_rows":
        int(
            Y_cal.shape[
                0
            ]
        ),

    "test_rows":
        int(
            Y_test.shape[
                0
            ]
        ),

    "labels":
        263,

    "best_global_threshold":
        float(
            BEST_GLOBAL_THRESHOLD
        ),

    "selected_threshold_method":
        SELECTED_METHOD,

    "calibration_global_objective":
        float(
            cal_global_objective
        ),

    "calibration_perlabel_objective":
        float(
            cal_perlabel_objective
        ),

    "test_metrics":
        {
            k:
                float(v)
            for k, v
            in primary_test_metrics.items()
        },

    "overall_calibration":
        {

            "Brier":
                float(
                    overall_brier
                ),

            "ECE":
                float(
                    overall_ece
                ),

            "NLL":
                float(
                    overall_nll
                )
        },

    "support_f1_spearman":
        float(
            support_f1_spearman
        ),

    "test_used_for_threshold_selection":
        False
}


SUMMARY_JSON = (
    OUT /
    "FINAL_ANALYSIS_SUMMARY.json"
)


with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 44) ZIP EVERYTHING
# ============================================================

ZIP_PATH = shutil.make_archive(

    "/kaggle/working/"
    "FINAL_UNIQUE263_PRIMARY_SIDE_EFFECT_ANALYSIS",

    "zip",

    root_dir=OUT
)


# ============================================================
# 45) FINAL RESULTS
# ============================================================

print("\n" + "=" * 120)
print("PRIMARY MODEL SIGNATURE")
print("=" * 120)

print(
    signature_df.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("PRIMARY FINAL TEST METRICS")
print("=" * 120)


for key, value in (
    primary_test_metrics.items()
):

    if key == "Hamming_Loss":

        print(
            f"{key:22s}: "
            f"{value:.8f}"
        )

    else:

        print(
            f"{key:22s}: "
            f"{100*value:.4f}%"
        )


print("\n" + "=" * 120)
print("FREQUENCY GROUPS")
print("=" * 120)


print(
    frequency_summary.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("THRESHOLD COMPARISON")
print("=" * 120)


print(

    threshold_comparison[
        [
            "Threshold_Strategy",
            "Calibration_Objective",
            "Test_Exact_Match",
            "Test_Micro_F1",
            "Test_Macro_F1",
            "Test_Weighted_F1",
            "Test_Hamming_Loss"
        ]
    ]

    .to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("PER-LABEL THRESHOLD SENSITIVITY")
print("=" * 120)


print(
    per_label_sensitivity.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("CALIBRATION")
print("=" * 120)


print(
    calibration_summary.to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("SPECIAL 3")
print("=" * 120)


print(

    special3[
        [
            "label_id",
            "label",
            "Train_Support",
            "Calibration_Support",
            "Test_Support",
            "Frequency_Group",
            "Selected_Threshold",
            "Precision",
            "Recall",
            "F1",
            "Brier_Score",
            "ECE_15bin"
        ]
    ]

    .to_string(
        index=False
    )
)


print("\n" + "=" * 120)
print("SUPPORT VS F1")
print("=" * 120)


print(
    "Spearman correlation:",
    support_f1_spearman
)


print("\n" + "=" * 120)
print("OUTPUT FILES")
print("=" * 120)


print(
    "Excel:"
)

print(
    XLSX_PATH
)


print(
    "\nZIP:"
)

print(
    ZIP_PATH
)


print("\n NO TRAINING PERFORMED")
print(" ORIGINAL baseline seeds 42 / 43 / 44")
print(" Equal ensemble")
print(" 263 per-label prevalence + Precision/Recall/F1")
print(" Rare / Medium / Frequent")
print(" Threshold comparison")
print(" Threshold sensitivity")
print(" Calibration analysis")
print(" Test set not used for threshold optimization")